# Laya PoC - Phase 1 smoke test (Colab T4 from VS Code)

Proves on a free Colab T4 that the single-GPU fine-tuning port works end to end before any paid run (design doc
§6.2, Phase 1): pinned install and GPU check, smoke data from FSQ, zero-shot on both shipped checkpoints, CPU-vs-GPU
parity, a 250-micro-step control run, a hard crash at micro-step 122 with a resume from the micro-step-120 checkpoint, an export/calibration round trip and a filled-in report.

**How to connect (VS Code + Google Colab extension)**
1. **Select Kernel → Colab → New Colab Server → GPU → T4** (shape *Standard*; runtime version *Latest*, or *2026.07*
   if offered), accept the alias, sign in to Google when asked, then pick the Python 3 kernel.
2. Do **not** use *Auto Connect*: it provisions a CPU server. If no T4 is free the extension may silently assign another
   GPU (watch for a toast). The runs still use the T4 batch profile, so the crash/resume plan holds, but VRAM and speed
   then do not describe a T4: prefer **Colab: Remove Server** and retry later. Step 4 records the card.
3. **Run All**, or run the steps in order. Keep VS Code open and the laptop awake: a running cell keeps the server alive.

**Hugging Face token:** Step 3 asks for it in a masked input box at the top of VS Code. It is never printed and never
stored in this notebook, only in the runtime's own token file, which is deleted with the server.

**After any kernel restart** (crash, manual restart, the optional kill drill): re-run **Step 1 (Setup)**, then continue
where you stopped. Finished steps are skipped because their outputs are on disk; set `REDO = True` in Step 1 to redo them.
Steps 9-10 are one drill: if Step 10 was interrupted, re-run Step 9, then Step 10.
A failing step names its log file; `docs/smoke_test_runbook.md` maps errors to fixes.

**Expected runtime:** about 60-90 min. Step 7's CPU fp32 parity reference is the longest step (about 25-35 min), then
the three training runs. Everything lives under `/content/laya_poc`; full logs are in `runs/smoke/logs/`.

**Exit criteria** (`config.yaml` `smoke.exit`, evaluated in Step 12):
- parity, each checkpoint: max |Δp| ≤ 0.02, argmax agreement ≥ 0.995, no NaN
- resume: the crash run is killed by the injected crash and the resumed run restarts from micro-step 120. Its
  mean `loss_ce` over the next 20 micro-steps is within 2% of the control run's, and the learning rates
  and GradScaler scale match the control run at every later optimiser step
- training: the control run's held-out val CE (eval mode) falls by ≥ 10% from the initial eval (opt step 0) to the
  final eval; no NaN/inf; GradScaler scale ≥ 1
- peak VRAM (reserved) ≤ 14.0 GB (10^9 bytes)

Code bundle: 59 files, sha256 `a2be6c79a792c739a8ad8027d46f45165aa7f05f686981178469eb2a3bbbe364` (verified when Step 1 unpacks it).


### Step 1 - Setup
Non-interactive environment, work dir, code bundle, helpers. Seconds. **Re-run this cell first after any kernel restart.**

In [ ]:
# Step 1 - Setup
import getpass, json, os, shutil, signal, subprocess, sys, time, zipfile
from pathlib import Path

os.environ.update({
    "COLAB_DISABLE_STDIN_FOR_SHELL_MAGICS": "1",  # a command that prompts gets EOF instead of hanging
    "GIT_TERMINAL_PROMPT": "0", "PIP_NO_INPUT": "1", "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1", "TOKENIZERS_PARALLELISM": "false",
    "LAYA_CUDA_AMP": "fp16",  # fp16 autocast on every card, also if the extension assigns an L4/A100
})
WORK = Path("/content/laya_poc")
DATA, RUNS = WORK / "data", WORK / "runs" / "smoke"
LOGS = RUNS / "logs"
for _d in (DATA, LOGS):
    _d.mkdir(parents=True, exist_ok=True)
os.environ["LAYA_POC_ROOT"] = str(WORK)  # the CLIs read WORK/config.yaml
REDO = False  # True: recompute steps whose outputs already exist
DRILL_MARK = Path.home() / ".laya_poc_kill_drill.json"  # outside WORK: Step 13d can tell a wiped /content


def write_json(path, obj) -> None:
    """Write JSON atomically, so a kill mid-write never leaves half a file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

def run_logged(cmd, log_path, expect_returncode=0, hint=None) -> int:
    """Run cmd with stdin closed (nothing can prompt); stream its output here and into log_path.

    VS Code truncates long cell output, so the log file is the full record. expect_returncode is an
    int, a tuple of ints, or None (accept any). Raises on an unexpected code; returns the code.
    """
    cmd, log_path = [str(c) for c in cmd], Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with open(log_path, "w", encoding="utf-8", buffering=1) as log:
        log.write("$ " + " ".join(cmd) + "\n")
        with subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace", bufsize=1, env=os.environ.copy()) as proc:
            try:
                for line in proc.stdout:
                    log.write(line)
                    print(line, end="", flush=True)
            except BaseException:  # interrupted cell: never leave an orphan holding the GPU
                proc.kill()
                raise
    rc = proc.returncode
    expected = (expect_returncode,) if isinstance(expect_returncode, int) else expect_returncode
    if expected is not None and rc not in expected:
        name = cmd[cmd.index("-m") + 1] if "-m" in cmd else Path(cmd[0]).name
        msg = f"{name} exited with {rc} (expected {' or '.join(map(str, expected))}); full log: {log_path}"
        raise RuntimeError(msg + (f"\nhint: {hint}" if hint else ""))
    return rc

def sh(cmd, check=True) -> str:
    """Run a short command with stdin closed; print and return its combined output."""
    res = subprocess.run(cmd, shell=isinstance(cmd, str), stdin=subprocess.DEVNULL, capture_output=True,
                         text=True, errors="replace")
    out = (res.stdout + res.stderr).strip()
    if out:
        print(out)
    if check and res.returncode:
        raise RuntimeError(f"command failed with exit code {res.returncode}: {cmd}")
    return out

def already_done(path) -> bool:
    """True (and says so) when a step's output exists and REDO is False, so re-runs skip finished steps."""
    if Path(path).exists() and not globals().get("REDO", False):
        print(f"skip: {path} exists (set REDO = True in Step 1 to recompute)")
        return True
    return False

def fresh_dir(path) -> Path:
    """Empty a run directory so a re-run starts from scratch instead of resuming a stale checkpoint."""
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True)
    return path

def show_json(path, keys=None, width=150):
    """Print one line per top-level key (only `keys` when any of them exist), without bulky per-row or
    per-class detail; return the whole object."""
    def brief(v):
        if isinstance(v, dict):
            return {k: brief(x) for k, x in v.items() if k not in ("per_row", "per_class", "cm", "confusion")}
        return v
    obj = json.loads(Path(path).read_text(encoding="utf-8"))
    shown = brief(obj) if isinstance(obj, dict) else {"value": obj}
    if keys and any(k in shown for k in keys):
        shown = {k: shown[k] for k in keys if k in shown}
    for key, val in shown.items():
        text = json.dumps(val, ensure_ascii=False, separators=(",", ":"))
        print(f"  {key}: {text if len(text) <= width else text[:width - 3] + '...'}")
    return obj

def show_markdown(path) -> None:
    """Render a markdown file in the notebook (plain text outside IPython, e.g. in the local dry run)."""
    text = Path(path).read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
    except ImportError:
        print(text)
        return
    display(Markdown(text))

def restore_hf_token() -> bool:
    """Put a previously saved HF token into os.environ for the CLIs. Never prints it."""
    if os.environ.get("HF_TOKEN"):
        return True
    try:
        from huggingface_hub import constants  # importing constants does not query the Colab vault
        token_file = Path(constants.HF_TOKEN_PATH)
    except ImportError:
        token_file = Path.home() / ".cache" / "huggingface" / "token"
    token = token_file.read_text(encoding="utf-8").strip() if token_file.exists() else ""
    if token:
        os.environ["HF_TOKEN"] = token
    return bool(token)

def resume_blocker(runs, kill_at=None):
    """Why Step 10 must not run now, or None. A resume is only valid straight after the crash run: the
    resumed log must end at the injected crash (at micro-step kill_at, when given). A -9 without it is some
    other kill, e.g. the host OOM killer. A second resume (interrupted Step 10, or REDO) would restart from
    a later checkpoint and the report, which grades the last 'resumed' event, would silently FAIL."""
    runs = Path(runs)
    exit_file, log_file = runs / "crash_exit.json", runs / "resumed" / "log.jsonl"
    if not exit_file.exists():
        return "no crash_exit.json: run Step 9 (crash run) first"
    rc = json.loads(exit_file.read_text(encoding="utf-8")).get("returncode")
    if rc not in (-9, 137):
        return f"the crash run exited with {rc}, not -9/137 (hard kill)"
    events = []
    for line in log_file.read_text(encoding="utf-8").splitlines() if log_file.exists() else []:
        try:
            event = json.loads(line)
        except ValueError:  # a SIGKILL can leave half a line
            continue
        if isinstance(event, dict):
            events.append(event)
    names = [e.get("event") for e in events]
    if "crash_injected" not in names:
        return "resumed/log.jsonl has no crash_injected event"
    if names[-1] != "crash_injected":
        return "a resume already ran (or was interrupted) after the crash"
    step = events[-1].get("micro_step")
    if kill_at is not None and step != kill_at:
        return f"the crash was injected at micro-step {step}, not at the planned {kill_at}"
    return None

def log_tail(path, n=8) -> str:
    """The last n lines of a log file, for error messages ('' when it does not exist)."""
    path = Path(path)
    if not path.exists():
        return ""
    return "\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-n:])

def gpu_pids() -> list[int]:
    """PIDs of processes holding GPU memory according to nvidia-smi; [] without an NVIDIA driver."""
    if not shutil.which("nvidia-smi"):
        return []
    out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], text=True,
                         capture_output=True, stdin=subprocess.DEVNULL).stdout
    return [int(x) for x in out.split() if x.isdigit()]

def pid_alive(pid) -> bool:
    """True while a process with this PID exists. Linux /proc only (the drill runs on Colab); False elsewhere,
    so nothing ever signals a PID just to probe it."""
    return Path(f"/proc/{int(pid)}").exists()

def proc_cmdline(pid) -> str:
    """The command line of a process in this PID namespace ('' when it is gone or not visible)."""
    try:
        return Path(f"/proc/{int(pid)}/cmdline").read_bytes().replace(b"\0", b" ").decode("utf-8", "replace")
    except OSError:
        return ""

def drill_orphans(gpu_pids_left, child_pid, marker="LAYA_POC_KILL_DRILL"):
    """(ours, others): ours are the kill drill's own GPU holders, recognised by the marker in their command
    line, and are safe to kill; never this kernel. nvidia-smi may report host-namespace PIDs, so any other
    PID is only reported: the same number can be an unrelated process in this container."""
    me = os.getpid()
    candidates = {int(p) for p in gpu_pids_left} | {int(child_pid)}
    ours = sorted(p for p in candidates if p != me and marker in proc_cmdline(p))
    others = sorted(p for p in {int(p) for p in gpu_pids_left} if p != me and p not in ours)
    return ours, others


SENTINEL = WORK / "sentinel.json"
if SENTINEL.exists():
    print("sentinel from", json.loads(SENTINEL.read_text(encoding="utf-8"))["created"], "found: /content survived")
else:
    write_json(SENTINEL, {"created": time.strftime("%Y-%m-%d %H:%M:%S"), "pid": os.getpid()})
    print("sentinel created: first Step 1 on this server (if Step 1 ran here before, /content was wiped)")
print("HF token: restored from the token file" if restore_hf_token() else "HF token: not set yet (Step 3)")

import base64, hashlib, io, pathlib, zipfile
def _unpack(b64, sha256, dest):
    raw = base64.b64decode(b64)
    assert hashlib.sha256(raw).hexdigest() == sha256, "bundle checksum mismatch"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for n in zf.namelist():
            assert dest.resolve() in (dest / n).resolve().parents, f"unsafe path {n}"
        zf.extractall(dest)
        return zf.namelist()
BUNDLE_SHA256 = "a2be6c79a792c739a8ad8027d46f45165aa7f05f686981178469eb2a3bbbe364"
BUNDLE_B64 = "..."
print(f"unpacked {len(_unpack(BUNDLE_B64, BUNDLE_SHA256, WORK))} files into {WORK}; bundle sha256 {BUNDLE_SHA256[:16]}")


### Step 2 - Install
Plain `laya` at the pinned commit, no extras (skipped when the PEP 610 record already matches), DuckDB, this project (editable, no deps). Fails if pip changed Colab's torch, transformers, protobuf or numpy. 1-3 min the first time.

In [ ]:
# Step 2 - Install
import importlib, importlib.metadata as md

LAYA_COMMIT = "4066d5d5fbf08b66c6757ddeedbd797bd7655bc0"
PIP = [sys.executable, "-m", "pip", "install", "-q", "--no-input", "--progress-bar", "off"]


def dist_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None


def direct_url(name):  # PEP 610 metadata: what pip actually installed, and from where
    try:
        return json.loads(md.distribution(name).read_text("direct_url.json") or "{}")
    except md.PackageNotFoundError:
        return {}


before = {p: dist_version(p) for p in ("torch", "transformers", "protobuf", "numpy")}  # Colab's own builds
if direct_url("laya").get("vcs_info", {}).get("commit_id") != LAYA_COMMIT:  # no [onnx] extra: it pulls protobuf 6+
    run_logged(PIP + [f"laya @ git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}"], LOGS / "02_install_laya.log")
duck = dist_version("duckdb")
if not duck or not (1, 1) <= tuple(int(x) for x in duck.split(".")[:2]) < (2, 0):
    run_logged(PIP + ["duckdb>=1.1,<2"], LOGS / "02_install_duckdb.log")
if not direct_url("laya-poc").get("dir_info", {}).get("editable"):
    run_logged(PIP + ["--no-deps", "-e", str(WORK)], LOGS / "02_install_project.log")
importlib.invalidate_caches()
changed = {p: f"{before[p]} -> {dist_version(p)}" for p in before if dist_version(p) != before[p]}
if changed:
    raise RuntimeError(f"pip changed {changed}; run Colab: Remove Server and start again")
got = direct_url("laya").get("vcs_info", {}).get("commit_id")
if got != LAYA_COMMIT:
    raise RuntimeError(f"installed laya commit {got} != pinned {LAYA_COMMIT}")
print({p: dist_version(p) for p in ("torch", "transformers", "huggingface_hub", "duckdb", "laya", "laya-poc")},
      "laya commit", got[:12])


### Step 3 - Hugging Face token
A masked input box opens at the top of VS Code. Paste a token that can read the gated FSQ dataset. It is validated, saved in the runtime's token file and never printed. To replace a saved token, set `ASK_AGAIN = True` in the cell. **Unattended run:** put `HF_TOKEN=hf_...` in the repo's `.env`, run `python tools/build_notebook.py --with-token` and open the gitignored `notebooks/smoke_test.local.ipynb` instead: no prompt.

In [ ]:
# Step 3 - Hugging Face token
import re
from huggingface_hub import login

PRESET_TOKEN = ""  # filled only in the gitignored smoke_test.local.ipynb (build_notebook.py --with-token)
ASK_AGAIN = False  # True: ignore the saved token (e.g. one without access to the gated dataset) and ask for a new one
NEW_TOKEN = "set ASK_AGAIN = True at the top of this cell and re-run it to enter a new token"
restore_hf_token()
token = os.environ.pop("HF_TOKEN", "").strip()
if PRESET_TOKEN and not ASK_AGAIN:  # unattended run: no prompt
    token = PRESET_TOKEN.strip()
del PRESET_TOKEN
if ASK_AGAIN or not token:  # VS Code shows a masked input box at the top of the window
    token = getpass.getpass("Hugging Face token (read access to foursquare/fsq-os-places; hidden): ").strip()
if not token:
    raise RuntimeError("no token entered (Esc/cancel sends an empty string); re-run this cell and paste it")
if not re.fullmatch(r"hf_[A-Za-z0-9]{20,}", token):  # never echo it: a line break would leak the rest in errors
    del token
    raise RuntimeError("that does not look like a Hugging Face token (hidden characters or line breaks?): "
                       + NEW_TOKEN)
try:
    login(token=token)  # validates with whoami and saves the token file, so a restart does not re-prompt
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    if status in (401, 403):
        raise RuntimeError(f"the Hub rejected the token (HTTP {status}): {NEW_TOKEN}") from None
    text = str(exc).replace(token, "***")[:200]  # the exact token first, then anything token-like
    detail = re.sub(r"hf_\w+", "hf_***", type(exc).__name__ + (f", HTTP {status}" if status else "") + f": {text}")
    raise RuntimeError(f"could not reach the Hub to check the token ({detail}); check the network and re-run this "
                       f"cell, or {NEW_TOKEN}") from None
os.environ["HF_TOKEN"] = token  # inherited by the CLIs (DuckDB reads the gated dataset with it)
del token
print("HF token OK (hidden)")


### Step 4 - Environment check
GPU, compute capability, `sm_75` in the CUDA arch list, driver, VRAM, free disk, versions and Laya commit -> `runs/smoke/env.json`. On failure it prints the errors from `env.json` (each names its fix; the runbook has one row per error). Warns if not a T4. ~30 s.

In [ ]:
# Step 4 - Environment check
if shutil.which("nvidia-smi"):
    sh(["nvidia-smi"], check=False)
env_json = RUNS / "env.json"
env_json.unlink(missing_ok=True)  # a stale file must not explain this run's failure
cmd = [sys.executable, "-m", "laya_poc.env_check", "--out", str(RUNS / "env.json"), "--require-gpu",
       "--expect-card", "T4"]
if run_logged(cmd, LOGS / "04_env.log", expect_returncode=None):  # each error in env.json names its fix
    errors = json.loads(env_json.read_text(encoding="utf-8")).get("errors") if env_json.exists() else None
    detail = "; ".join(errors) if errors else f"no env.json written; see {LOGS / '04_env.log'}"
    raise RuntimeError(f"environment check failed: {detail} (runbook: When something fails)")
env = show_json(env_json, ["gpu_name", "capability", "has_sm75", "driver", "vram_total_gb", "disk_free_gb", "torch", "transformers", "laya_commit", "card", "warnings"])


### Step 5 - Smoke data
DuckDB over `hf://` (one best places file per ID country) -> splits, JSONL, leakage asserts -> `data/`. 2-5 min.

In [ ]:
# Step 5 - Smoke data
if not already_done(DATA / "SHA256SUMS"):
    cmd = [sys.executable, "-m", "laya_poc.build_data", "--smoke", "--out", str(DATA)]
    run_logged(cmd, LOGS / "05_data.log",
               hint="401/403: accept the terms of the gated dataset foursquare/fsq-os-places on the Hub; for a different token set ASK_AGAIN = True in Step 3 and re-run it; then re-run this step")
data_report = show_json(DATA / "data_report.json", ["split_sizes", "budgets", "warnings"])


### Step 6 - Zero-shot
Both shipped checkpoints on the first 20 labelled val rows (shipped temperatures). 3-6 min including the checkpoint downloads.

In [ ]:
# Step 6 - Zero-shot
if not already_done(RUNS / "zeroshot.json"):
    cmd = [sys.executable, "-m", "laya_poc.zeroshot", "--models", "laya", "laya_ml",
           "--rows", str(DATA / "val.jsonl"), "--n", "20", "--out", str(RUNS / "zeroshot.json"),
           "--device", "cuda"]
    run_logged(cmd, LOGS / "06_zeroshot.log")
zeroshot = show_json(RUNS / "zeroshot.json")


### Step 7 - Parity
CPU fp32 reference vs GPU fp16 on 200 val rows, plus padded vs unpadded batches, for each checkpoint. Prints PASS/FAIL. About 25-35 min, the longest step: the fp32 reference runs on Colab's CPU while the T4 waits.

In [ ]:
# Step 7 - Parity
if not already_done(RUNS / "parity_laya.json"):
    cmd = [sys.executable, "-m", "laya_poc.parity", "--model", "laya", "--rows", str(DATA / "val.jsonl"),
           "--n", "200", "--out", str(RUNS / "parity_laya.json")]
    run_logged(cmd, LOGS / "07_parity_laya.log")
if not already_done(RUNS / "parity_laya_ml.json"):
    cmd = [sys.executable, "-m", "laya_poc.parity", "--model", "laya_ml", "--rows", str(DATA / "val.jsonl"),
           "--n", "200", "--out", str(RUNS / "parity_laya_ml.json")]
    run_logged(cmd, LOGS / "07_parity_laya_ml.log")
for m in ("laya", "laya_ml"):
    print("parity", m)
    parity = show_json(RUNS / f"parity_{m}.json", ["max_dp", "argmax_agree", "padded_max_dp", "nan", "passed", "device_test", "dtype_test"])


### Step 8 - Control run
250 micro-steps with an initial eval (val CE at opt step 0), a final eval and a final checkpoint; no resumable checkpoints (nothing reads them, and the crash run measures the save time). Steps 8-10 use the T4 batch profile (`--card T4`: micro-batch 8 x accumulation 4) on any GPU, so the plan holds. 5-10 min.

In [ ]:
# Step 8 - Control run
if not already_done(RUNS / "control" / "summary.json"):
    fresh_dir(RUNS / "control")
    cmd = [sys.executable, "-m", "laya_poc.train_single", "--run-dir", str(RUNS / "control"),
           "--data-dir", str(DATA), "--model", "laya", "--seed", "11", "--epochs", "1", "--device", "cuda",
           "--card", "T4", "--max-micro-steps", "250", "--ckpt-every-micro-steps", "0",
           "--ckpt-every-min", "0", "--initial-eval", "--final-eval", "--save-final"]
    run_logged(cmd, LOGS / "08_control.log",
               hint="CUDA OOM: add '--micro-batch', '4', '--effective-batch', '32' to cmd here and in Steps 9-10 (runbook: When something fails)")
control = show_json(RUNS / "control" / "summary.json", ["micro_steps", "opt_steps", "resumed_from", "peak_vram_reserved_gb", "sec_per_micro_median", "nonfinite", "min_scale", "initial_eval", "final_eval"])


### Step 9 - Crash run
Same settings with a resumable checkpoint every 40 micro-steps, hard-killed (SIGKILL) right after micro-step 122; exit code -9/137 goes to `crash_exit.json`, and `resumed/log.jsonl` must end with that injected crash (any other kill, e.g. the host OOM killer, stops here). Starts the drill afresh. 3-5 min.

In [ ]:
# Step 9 - Crash run
if not already_done(RUNS / "resumed" / "summary.json"):
    fresh_dir(RUNS / "resumed")
    (RUNS / "crash_exit.json").unlink(missing_ok=True)
    cmd = [sys.executable, "-m", "laya_poc.train_single", "--run-dir", str(RUNS / "resumed"),
           "--data-dir", str(DATA), "--model", "laya", "--seed", "11", "--epochs", "1", "--device", "cuda",
           "--card", "T4", "--max-micro-steps", "250", "--ckpt-every-micro-steps", "40",
           "--crash-at-micro-step", "122"]
    rc = run_logged(cmd, LOGS / "09_crash.log", expect_returncode=None)
    write_json(RUNS / "crash_exit.json", {"returncode": rc})
    if rc not in (-9, 137):
        raise RuntimeError(f"crash run exited with {rc}, expected -9 or 137 (hard kill); see {LOGS / '09_crash.log'}")
    problem = resume_blocker(RUNS, 122)  # the kill must be the injected crash, not e.g. the OOM killer
    if problem:
        raise RuntimeError(f"the crash run was killed ({rc}) but not as planned: {problem}. A kill without "
                           "crash_injected is usually the host OOM killer: check RAM (free -h) and the kernel log "
                           f"(dmesg | tail), then re-run this step. Last lines of {LOGS / '09_crash.log'}:\n"
                           + log_tail(LOGS / "09_crash.log"))
    print("crash run was hard-killed as planned at micro-step 122; exit code", rc)


### Step 10 - Resume
The crash run's settings without the crash flag, and with no checkpoints (nothing reads them), must print `resumed from ...` (micro-step 120) and finish. Runs once per crash: if it was interrupted, re-run Step 9 first. 3-5 min.

In [ ]:
# Step 10 - Resume
if not already_done(RUNS / "resumed" / "summary.json"):
    problem = resume_blocker(RUNS, 122)  # one resume per crash, else the report grades another
    if problem:
        raise RuntimeError(f"the crash/resume drill is incomplete ({problem}): re-run Step 9 (with "
                           "REDO = True in Step 1 if Step 10 had finished), then Step 10")
    cmd = [sys.executable, "-m", "laya_poc.train_single", "--run-dir", str(RUNS / "resumed"),
           "--data-dir", str(DATA), "--model", "laya", "--seed", "11", "--epochs", "1", "--device", "cuda",
           "--card", "T4", "--max-micro-steps", "250", "--ckpt-every-micro-steps", "0",
           "--ckpt-every-min", "0"]
    run_logged(cmd, LOGS / "10_resume.log")
    if "resumed from" not in (LOGS / "10_resume.log").read_text(encoding="utf-8"):
        raise RuntimeError("no 'resumed from ...' line: the run restarted from scratch")
resumed = show_json(RUNS / "resumed" / "summary.json", ["micro_steps", "opt_steps", "resumed_from", "peak_vram_reserved_gb", "sec_per_micro_median", "nonfinite", "min_scale", "initial_eval", "final_eval"])


### Step 11 - Export/calibration round trip
Fit T on raw val logits, write it into the saved checkpoint, reload with Laya, compare probabilities. 1-2 min.

In [ ]:
# Step 11 - Export/calibration round trip
if not already_done(RUNS / "export_check.json"):
    cmd = [sys.executable, "-m", "laya_poc.export_check", "--ckpt", str(RUNS / "control" / "final"),
           "--rows", str(DATA / "val.jsonl"), "--out", str(RUNS / "export_check.json"), "--device", "cuda"]
    run_logged(cmd, LOGS / "11_export_check.log")
export = show_json(RUNS / "export_check.json", ["T", "clamped", "roundtrip_max_dp", "passed"])


### Step 12 - Report
Exit criteria and the design doc §7.13 template -> `runs/smoke/smoke_report.md`.

In [ ]:
# Step 12 - Report
cmd = [sys.executable, "-m", "laya_poc.smoke_report", "--run-root", str(RUNS), "--data-dir", str(DATA),
       "--out", str(RUNS / "smoke_report.md")]
run_logged(cmd, LOGS / "12_report.log")
show_markdown(RUNS / "smoke_report.md")


### Step 13a - Archive (optional)
---
Optional steps; each is switched by a flag in its cell. Zip the small artefacts and the FSQ NOTICE (`data/NOTICE_FSQ.txt`) for **Download...** in the Colab view; optional Drive copy.

In [ ]:
# Step 13a - Archive (optional)
MAKE_ARCHIVE = True    # logs, JSON, the report and the FSQ NOTICE only: no checkpoints, no FSQ rows
COPY_TO_DRIVE = False  # interactive Google sign-in (about 2 min); fails for some Workspace accounts
ARCHIVE = WORK / "smoke_artifacts.zip"
if MAKE_ARCHIVE:
    small = {".json", ".jsonl", ".md", ".log", ".yaml"}
    keep = [p for p in sorted(RUNS.rglob("*"))
            if p.is_file() and p.suffix in small and not {"ckpt", "final"} & set(p.relative_to(RUNS).parts)]
    keep += [p for p in (DATA / "data_report.json", DATA / "SHA256SUMS", DATA / "NOTICE_FSQ.txt") if p.exists()]
    with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in keep:
            zf.write(p, p.relative_to(WORK).as_posix())
    print(f"{ARCHIVE}: {len(keep)} files; Colab view > Contents > right-click > Download...")
if COPY_TO_DRIVE and ARCHIVE.exists():
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/laya_poc")
    dest.mkdir(parents=True, exist_ok=True)
    print("copied to", shutil.copy2(ARCHIVE, dest / ARCHIVE.name))


### Step 13b - Free disk (optional)
Delete the resumable training checkpoints.

In [ ]:
# Step 13b - Free disk (optional)
CLEANUP_CKPTS = False  # True: delete the resumable checkpoints (about 5 GB each); final/ is kept
if CLEANUP_CKPTS:
    for ckpt_dir in sorted(RUNS.glob("*/ckpt")):
        size = sum(f.stat().st_size for f in ckpt_dir.rglob("*") if f.is_file())
        shutil.rmtree(ckpt_dir)
        print(f"removed {ckpt_dir} ({size / 1e9:.1f} GB)")
else:
    print("checkpoint cleanup skipped (CLEANUP_CKPTS = False)")


### Step 13c - Kernel-kill drill (optional, off)
Hard-kills the kernel while a child holds the GPU. Afterwards re-run Step 1, then Step 13d.

In [ ]:
# Step 13c - Kernel-kill drill (optional, off)
KILL_DRILL = False  # True: hard-kill this kernel (VS Code reports a crash - expected), then re-run Step 1 and 13d
if KILL_DRILL:
    # keeps a CUDA context; the marker lets Step 13d recognise (and only ever kill) this process
    holder = "import time, torch; LAYA_POC_KILL_DRILL = 1; torch.ones(1, device='cuda'); time.sleep(900)"
    child = subprocess.Popen([sys.executable, "-c", holder],
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(60):  # wait until the child holds a CUDA context (the kernel itself never touches the GPU)
        if gpu_pids() or child.poll() is not None:
            break
        time.sleep(1)
    created = json.loads(SENTINEL.read_text(encoding="utf-8"))["created"] if SENTINEL.exists() else None
    record = {"kernel_pid": os.getpid(), "child_pid": child.pid, "gpu_pids": gpu_pids(), "sentinel_created": created,
              "time": time.strftime("%Y-%m-%d %H:%M:%S")}
    for path in (RUNS / "kill_drill.json", DRILL_MARK):
        write_json(path, record)
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("kernel-kill drill skipped (KILL_DRILL = False)")


### Step 13d - After the kill drill
Checks that `/content` and site-packages survived and that no orphan process is left on the GPU (kills only the drill's own child).

In [ ]:
# Step 13d - After the kill drill
import importlib.util

drill = RUNS / "kill_drill.json"
record = drill if drill.exists() else DRILL_MARK if DRILL_MARK.exists() else None
if record is None:
    print("no kill drill recorded (run Step 13c with KILL_DRILL = True first)")
else:
    d, left = json.loads(record.read_text(encoding="utf-8")), gpu_pids()
    sentinel = json.loads(SENTINEL.read_text(encoding="utf-8")) if SENTINEL.exists() else {}
    ours, others = drill_orphans(left, d["child_pid"])
    checks = {"kernel restarted": d["kernel_pid"] != os.getpid(),
              "/content survived (drill record and the original sentinel)":
                  drill.exists() and sentinel.get("created") == d.get("sentinel_created"),
              "site-packages survived (laya importable)": importlib.util.find_spec("laya") is not None,
              "GPU child died with the kernel": not pid_alive(d["child_pid"]),
              "no process left on the GPU": not left}
    write_json(RUNS / "kill_drill_result.json", {**checks, "gpu_pids_after": left, "drill": d})
    for name, ok in checks.items():
        print("PASS" if ok else "FAIL", name)
    for pid in ours:
        try:
            os.kill(pid, signal.SIGKILL)
            print("killed the drill's orphan", pid)
        except OSError as exc:
            print("could not kill", pid, exc)
    if others:
        print("other GPU processes, not started by the drill (left alone):", others)


### Finish
- The verdict is at the end of Step 12; details in `runs/smoke/smoke_report.md` and `smoke_report.json`.
- Download `smoke_artifacts.zip` (Step 13a): Colab view in the activity bar > Contents > right-click > **Download...**
- Run **Colab: Remove Server** when you are done. It stops quota use and deletes `/content`, checkpoints included.
